# InstructionAdherenceEvaluator usage

Instruction adherence asks whether explicit output instructions were followed. It requires `instructions + output`; `context` is optional supporting evidence and never creates new instructions. Optional `input` is descriptive and is not sent to the Instruction Adherence judge. One case makes one judge call.

In [ ]:
from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    InstructionAdherenceEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## Configure a judge

Use placeholders only. Production applications normally populate this config from their settings/secrets layer. Evaluators are backend-independent; `create_gateway_judge(config=gateway_config)` can be used instead. See the setup guide and backend latency notebook for backend configuration.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

A plain multiline string is the common production instruction shape. Dict, list, and nested structured instructions are also supported through `render_value()`. A list in `case.output` is one complete structured output. Context below is evidence only for the explicit approved-options instruction.

In [ ]:
instructions = """
Generate exactly 3 options.
Every option must contain a title.
Do not include implementation details.
Only use approved options from the supplied context.
"""

case = EvaluationCase(
    input="Generate approved deployment options.",
    instructions=instructions,
    context={
        "approved_options": ["Option A", "Option B", "Option C"]
    },
    output=[
        {"title": "Option A"},
        {"title": "Option B"},
        {"title": "Option C"},
    ],
)
evaluator = InstructionAdherenceEvaluator(judge, verbose=True)
framework = EvaluationFramework(
    judge=judge,
    evaluators=[evaluator],
)
result = framework.evaluate(case)["instruction_adherence"]
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "details": result.details,
}

## Optional async equivalent

Jupyter supports top-level `await`; this is an alternative execution example and reuses the same setup.

In [ ]:
async_result = await framework.a_evaluate(case)
async_result["instruction_adherence"]

In [ ]:
judge.close()